[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/CI-COURSE/blob/main/Colab_3_Advanced_Resource_Management.ipynb)







# 🧭 Colab 3: Advanced Neuro-Workflows and Resource Management



## Scientific Workflows for Brain and Behavioral Research



### National Science Foundation Supported Learning Initiative (Award No. OAC-2417875)



#### Semester Course — Colab 3 of 5 (Weeks 5–6)







***







## Overview







An enlarged version of Notebook 03 (FABRIC/CI Access) and the onsite workshop's Lab Stage 3 (EdgeBMI) — extended from a single simulated micro:bit exchange into real multi-node FABRIC scheduling and a genuine two-way edge-device flow.







Split into two submissions:



- **Colab 3-1** (end of Week 5): a brain-circuit model distributed across multiple FABRIC nodes.



- **Colab 3-2** (end of Week 6, combined): a two-way edge-device task of your team's choosing, plus capstone team selection.







***



## 📋 Set Up Your Profile

In [ ]:
# PROFILE SETUP



student_name = "Rowan Tate"



institution = "University of Missouri"



team_name = "CS 4001"



print(f"✅ Notebook initialized for {student_name} ({institution}) — Team {team_name}")

***
## 🪜 Your Cumulative Microcredential Ladder — Colab 3 Additions

Copy your Ladder forward from Colab 2. This Colab introduces an entire new ladder — Compute Access Model — built directly on the Slurm content you're about to learn. Every rung needs a real number.

***



## 🖥️ Part A (Primer): Advanced Neuro-Workflows and Resource Management



### Topic: Job Scheduling, Slurm, Multi-Node Jobs







FABRIC (and most shared research computing) uses a **job scheduler** — commonly Slurm — to decide which queued jobs run where and when. This section introduces the vocabulary and habits that go with it.







### 🧭 Activity A.1 — Writing a Batch Job Request







> "Act as a research-computing instructor teaching Slurm to a student who has only used interactive sessions before. Write an example Slurm batch script (using `#SBATCH` directives) that requests 4 nodes, 8 cores per node, 16 GB memory, and a 2-hour wall-clock time limit for a research analysis job. Add comments explaining what each directive controls and what happens if the job exceeds its time limit."

In [ ]:
# 💻 Activity A.1 — Paste your AI-generated Slurm batch-script example here:

Below is a typical Slurm batch script for a multi-node research analysis job. I’m assuming **16 GB of memory per node**, which is the usual interpretation when using `--mem=16G`.

```bash
#!/bin/bash

# Give the job a recognizable name
#SBATCH --job-name=research_analysis

# Request 4 compute nodes
#SBATCH --nodes=4

# Run 8 tasks/cores on each node
# 4 nodes × 8 cores = 32 CPU cores total
#SBATCH --ntasks-per-node=8

# Request 16 GB of RAM on EACH node
# With 4 nodes, this can allocate up to 64 GB total
#SBATCH --mem=16G

# Maximum wall-clock runtime: 2 hours
#SBATCH --time=02:00:00

# Save normal program output to this file
# %j is replaced with the Slurm job ID
#SBATCH --output=research_%j.out

# Save error messages to this file
#SBATCH --error=research_%j.err


# -----------------------------
# Commands executed by the job
# -----------------------------

echo "Job started on: $(date)"
echo "Job ID: $SLURM_JOB_ID"
echo "Number of nodes: $SLURM_NNODES"

# Load any software needed for the analysis.
# The exact module will depend on your cluster.
# module load python

# Example:
# python research_analysis.py

# For a parallel program that can use all 32 tasks,
# you would commonly launch it with srun:
srun ./research_analysis

echo "Job finished on: $(date)"
```

The important difference from an interactive session is that you **submit this script to Slurm instead of immediately receiving a shell**:

```bash
sbatch research_job.sh
```

Slurm puts the request in a queue, waits until the requested resources are available, and then runs the script automatically.

The resource request can be summarized as:

| Directive | Meaning |
|---|---|
| `--nodes=4` | Allocate 4 compute nodes |
| `--ntasks-per-node=8` | Use 8 CPU tasks/cores on each node |
| `--mem=16G` | Allocate 16 GB RAM **per node** |
| `--time=02:00:00` | Allow the job to run for at most 2 hours |
| `--output=...` | File containing normal output |
| `--error=...` | File containing error messages |

So the job has **4 nodes × 8 cores = 32 CPU cores total** and, with this script, up to **4 × 16 GB = 64 GB RAM total**.

### What happens after 2 hours?

`--time=02:00:00` is a **hard wall-clock limit**, not an estimate. If the analysis finishes in 45 minutes, the job ends normally and the unused time is simply released. If it is still running when it reaches **2 hours**, Slurm terminates the job. Any results that were only stored in RAM and had not yet been written to disk can therefore be lost.

That is one reason long research programs often periodically save **checkpoints or intermediate results**.

One important detail: if your assignment means **16 GB total across all four nodes**, rather than 16 GB per node, you would generally request approximately:

```bash
#SBATCH --mem=4G
```

per node, giving about `4 nodes × 4 GB = 16 GB` overall. Cluster policies can vary, but `--mem` normally specifies memory **per node**.

### 🧭 Activity A.2 — Monitoring and Cleaning Up a Job







> "Write example commands (or a short Python wrapper) for checking a submitted job's state (Pending / Running / Completed / Failed), viewing its standard output/error log, and cancelling it early if something looks wrong. Add comments explaining why monitoring matters on a shared cluster."

In [ ]:
# 💻 Activity A.2 — Paste your AI-generated job-monitoring code here:

Here’s a compact set of Slurm commands you can use after submitting a job with `sbatch`.

```bash
# Submit the job.
# Slurm will return a job ID, for example: "Submitted batch job 12345"
sbatch research_job.sh

# Check whether the job is Pending (PD) or Running (R).
# Replace 12345 with your actual job ID.
squeue -j 12345

# Show more detailed information about the job,
# including why it may still be pending.
scontrol show job 12345

# After the job finishes, check its final state.
# Common states include COMPLETED, FAILED, CANCELLED, and TIMEOUT.
sacct -j 12345 --format=JobID,JobName,State,Elapsed,ExitCode
```

If your batch script used something like:

```bash
#SBATCH --output=research_%j.out
#SBATCH --error=research_%j.err
```

then `%j` becomes the job ID. For job `12345`, you can inspect the logs with:

```bash
# View standard output
cat research_12345.out

# View error messages
cat research_12345.err

# Follow output live while the job is running
tail -f research_12345.out
```

If you notice a serious problem, such as repeated errors, incorrect input data, or runaway resource usage, cancel the job with:

```bash
scancel 12345
```

You can also check several of your jobs at once:

```bash
# Show all of your currently queued/running jobs
squeue -u $USER
```

A typical monitoring workflow would therefore be:

```bash
# 1. Submit
sbatch research_job.sh

# 2. Check queue state
squeue -j 12345

# 3. Watch the output
tail -f research_12345.out

# 4. Check final result after it leaves the queue
sacct -j 12345 --format=JobID,State,Elapsed,ExitCode

# 5. Cancel if something is clearly wrong
scancel 12345
```

Monitoring matters on a shared cluster because CPU cores, memory, GPUs, and compute nodes are shared resources. A broken job that keeps running can waste your allocation and prevent other researchers from using those resources. Checking the job state and logs lets you catch problems early instead of discovering hours later that the job was using the wrong input, repeatedly crashing, or producing invalid results.

The most common Slurm states you’ll see are:

| State | Meaning |
|---|---|
| `PD` / `PENDING` | Waiting for resources or scheduling |
| `R` / `RUNNING` | Currently executing |
| `COMPLETED` | Finished successfully |
| `FAILED` | Program exited unsuccessfully |
| `CANCELLED` | Job was manually or automatically cancelled |
| `TIMEOUT` | Job exceeded its requested wall-clock limit |

One detail that often confuses beginners: `squeue` mainly shows jobs that are still active, while `sacct` is usually what you use to inspect a job **after it has finished**.

### 🧭 Activity A.3 — Vibe-Coding HW: Exploring Globus on Your Own







So far in this Colab, you've worked with Slurm to manage *where and when* computations run. But managing a distributed workflow isn't just about compute time — it's also about getting data to and from the resources doing the work. That's what **Globus** is for.







Globus is a research data-management service used to move large datasets reliably between institutions and computing resources (e.g., from your lab's storage to a FABRIC node, or between two different national computing systems) — without a transfer failing partway through and silently corrupting the data.







As with Colab 2's CloudLab/NSG exploration, there's **no starting prompt here**. Use an AI tool to teach yourself:







1. What problem Globus actually solves that a simple file-copy command doesn't. (Hint: think about what happens if a multi-gigabyte transfer fails halfway through.)



2. How a researcher sets up and runs a basic transfer between two "endpoints" in Globus.



3. Write (and run, if you have access; otherwise write as clearly-commented pseudocode) a short script or step-by-step procedure that would transfer a dataset from one location to another using Globus.







As before, **keep a short log of your own prompt iterations** — what you asked first, what you had to correct or clarify, and how your understanding changed.







**Connecting back to Colab 3:** After completing this, briefly note — if your Activity B.1 multi-node circuit model needed to pull its input data from an external source before running, which step of your workflow would Globus fit into?

In [ ]:
# 💻 Activity A.3 — Paste your AI-generated Globus exploration code/procedure here, and your prompt-iteration log:
 What problem Globus actually solves that a simple file-copy command doesn't. Walk me through some examples:
Globus solves the problem of moving **large research datasets reliably between computers, clusters, and storage systems**. A normal command like `cp` or `scp` works well for small transfers, but it becomes inconvenient when transfers are large, long-running, or cross multiple research systems.

Here are a few concrete examples.

### Example 1: A transfer fails halfway through

Suppose you are copying a **200 GB EEG dataset** from your laptop to a research cluster using:

```bash
scp eeg_dataset.tar user@cluster:/data/
```

If your Wi-Fi disconnects after 150 GB, the transfer may fail and you may have to restart or manually figure out what was successfully copied.

With Globus, the transfer is managed as a job. If the connection temporarily fails, Globus can **retry the transfer automatically** rather than expecting you to restart everything manually.

So the difference is roughly:

```text
scp:
Start → transfer → network failure → transfer stops

Globus:
Start → transfer → network failure → retry → continue → finish
```

---

### Example 2: Thousands of files

Imagine your EEG project contains:

```text
GAMEEMO/
├── S01/
│   ├── game1.csv
│   ├── game2.csv
│   └── ...
├── S02/
├── S03/
...
└── S28/
```

There might be thousands of files.

You could use:

```bash
scp -r GAMEEMO user@cluster:/data/
```

but if something goes wrong, determining which files successfully transferred can become difficult.

Globus tracks the files in the transfer and can verify that they arrived correctly. It can also use **checksums** to detect corrupted files.

For example:

```text
Source file
EEG_S01.csv
      ↓
Globus transfers file
      ↓
checksum verification
      ↓
Destination file confirmed correct
```

---

### Example 3: Moving data between two supercomputers

Suppose your dataset is stored at one university:

```text
University A HPC
        ↓
     2 TB data
        ↓
University B HPC
```

Using `scp`, your computer may effectively be involved in managing the session, and you may need to keep terminals, credentials, or sessions active.

With Globus, you tell the Globus service something like:

```text
Transfer:

University A storage
        ↓
University B storage
```

The systems transfer the data directly between their **Globus endpoints/collections**.

Your laptop does not need to sit there managing the transfer.

You could close your browser or shut down your laptop while the transfer continues.

---

### Example 4: A transfer takes several hours

Suppose you have a **500 GB neuroscience dataset**.

You start:

```bash
scp dataset.tar cluster:/data/
```

If the transfer takes 8 hours, you might need to worry about:

- your SSH connection dropping
- your laptop sleeping
- Wi-Fi disconnecting
- authentication timing out
- restarting failed transfers

With Globus, you submit the transfer and Globus manages those details for you.

Conceptually:

```text
You:
"Move Dataset A from Collection X to Collection Y."

Globus:
✓ starts transfer
✓ retries failures
✓ verifies files
✓ tracks progress
✓ records transfer status
✓ reports when finished
```

---

### Example 5: Collaborating with another researcher

Suppose another university has a **1 TB MRI dataset** you need.

Instead of the researcher giving you an SSH account and telling you to run:

```bash
scp researcher@university.edu:/data/mri/* .
```

they can share a Globus collection with you.

You can then transfer:

```text
Collaborator's Globus Collection
              ↓
      Your HPC Collection
```

This is particularly useful because researchers can control **which folders you are allowed to access** without necessarily giving you full login access to their server.

---

### The key problem Globus solves

Think of `scp` as:

> "Copy these files right now."

Globus is closer to:

> "Make sure these files successfully get from System A to System B, even if the transfer takes hours or encounters temporary failures."

For research computing, its biggest advantages are **reliability, automatic retries, integrity checking, large-data support, transfer monitoring, and the ability to move data directly between research systems**.

For something like your EEG workflow, a realistic use case might be:

```text
Laptop / Local Storage
        ↓
      Globus
        ↓
University HPC / ACCESS / FABRIC storage
        ↓
Run EEG preprocessing or CNN training
```

Instead of manually babysitting a huge `scp` command, Globus handles the transfer as a managed task.


Explain and provide an example of How a researcher sets up and runs a basic transfer between two "endpoints" in Globus.

A researcher usually thinks of a Globus transfer as moving data between **two storage locations**. Globus often calls these **collections**; older tutorials and casual explanations may call them **endpoints**. Technically, an endpoint represents the server/system, while a collection is the actual location you use to browse and transfer files. [Globus Docs](https://docs.globus.org/guides/overviews/collections-and-endpoints/?utm_source=chatgpt.com)

Here’s a typical setup.

### Example: Transfer EEG data from a laptop to a university cluster

Suppose you have this dataset on your laptop:

```text
C:\Research\GAMEEMO\
```

and you want it on your university HPC cluster:

```text
/home/rowan/EEG_Data/
```

The flow would look like:

```text
Your Laptop
Globus Connect Personal
       │
       │   Globus transfer
       ▼
University HPC Cluster
Globus Connect Server
```

Globus Connect Personal is specifically intended to make a laptop or personal desktop accessible to Globus, while institutions typically use Globus Connect Server for shared servers and HPC systems. [Globus Docs](https://docs.globus.org/faq/globus-connect-endpoints/?utm_source=chatgpt.com)

### 1. Log into Globus

Go to the Globus web application and sign in, often using your university account.

After logging in, open **File Manager**.

You will normally see two file-browser panels:

```text
SOURCE                         DESTINATION
--------------------           --------------------
My Laptop                      University HPC
GAMEEMO/                       /home/rowan/
```

The two panels make it easy to select where data is coming from and where it should go. Globus's official getting-started tutorial uses this same File Manager approach. [Globus Docs](https://docs.globus.org/guides/tutorials/manage-files/transfer-files/?utm_source=chatgpt.com)

### 2. Make your laptop accessible to Globus

If the source is your laptop, install **Globus Connect Personal**.

During setup, you create a personal Globus collection, perhaps named:

```text
Rowan-Laptop
```

Globus Connect Personal then allows Globus to transfer files to and from that computer. [Globus Docs](https://docs.globus.org/faq/globus-connect-endpoints/?utm_source=chatgpt.com)

For example:

```text
Collection:
Rowan-Laptop

Folder:
C:\Research\GAMEEMO\
```

You generally leave Globus Connect Personal running while files are being transferred to or from that computer.

### 3. Find the cluster's Globus collection

The university or research center usually sets this side up for you.

For example, imagine the institution provides a collection called:

```text
University HPC Storage
```

In Globus File Manager you search for it and authenticate if required.

Now you have:

```text
LEFT SIDE                        RIGHT SIDE

Rowan-Laptop                     University HPC Storage
C:\Research\GAMEEMO\             /home/rowan/EEG_Data/
```

On multi-user systems, administrators normally provide this access through **Globus Connect Server**. [Globus Docs](https://docs.globus.org/faq/globus-connect-endpoints/?utm_source=chatgpt.com)

### 4. Select the dataset

On the laptop side, select:

```text
GAMEEMO/
```

For example:

```text
GAMEEMO/
├── S01/
│   ├── Game1.csv
│   ├── Game2.csv
│   ├── Game3.csv
│   └── Game4.csv
├── S02/
├── S03/
...
└── S28/
```

On the HPC side, navigate to:

```text
/home/rowan/EEG_Data/
```

Then click **Start** or the transfer arrow.

Conceptually you are telling Globus:

```text
SOURCE
Rowan-Laptop:/Research/GAMEEMO/

        ↓ TRANSFER

DESTINATION
University-HPC:/home/rowan/EEG_Data/
```

### 5. Globus creates a transfer task

At this point, you don't need to manually copy every file.

Globus submits the operation as an **asynchronous transfer task**, meaning Globus manages and monitors the transfer rather than requiring you to continuously run a command in a terminal. [Globus Docs](https://docs.globus.org/api/transfer/?utm_source=chatgpt.com)

You might see something like:

```text
Transfer submitted successfully.

Source:
Rowan-Laptop:/Research/GAMEEMO/

Destination:
University-HPC:/home/rowan/EEG_Data/

Status:
ACTIVE
```

You can then check the **Activity** section.

Eventually:

```text
Status: SUCCEEDED
```

Globus can manage retries and track progress for the task, which is one of the major benefits compared with simply launching a long `scp` session. [Globus Docs](https://docs.globus.org/api/transfer/?utm_source=chatgpt.com)

### 6. Use the data on the cluster

After the transfer finishes, you might SSH into the cluster:

```bash
ssh rowan@hpc.university.edu
```

and verify the files:

```bash
cd /home/rowan/EEG_Data
ls
```

You might see:

```text
S01
S02
S03
...
S28
```

Then your Python program or Slurm job can access the data normally:

```python
import numpy as np

data = np.load("/home/rowan/EEG_Data/eeg_data.npy")
```

Globus is only handling the **movement of the data**. Your research program doesn't have to use Globus to actually read the files afterward.

### Complete workflow

For a research workflow like your EEG/CNN project, the entire process could be:

```text
1. Collect/download EEG dataset
             │
             ▼
2. Data stored on laptop
   C:\Research\GAMEEMO
             │
             ▼
3. Run Globus Connect Personal
             │
             ▼
4. Open Globus File Manager
             │
             ▼
5. Source:
   Rowan-Laptop
             │
             │
       Globus transfer
             │
             ▼
6. Destination:
   University HPC Storage
             │
             ▼
7. /home/rowan/EEG_Data/
             │
             ▼
8. Submit Slurm job
             │
             ▼
9. Train CNN on EEG data
```

So, in one sentence:

> **A researcher identifies or creates a Globus collection at the source and destination, opens both in Globus File Manager, selects the desired files and destination directory, starts the transfer, and then lets Globus manage and monitor the transfer until it completes.**

One important terminology note for an assignment: saying **"transfer between two endpoints"** is usually understandable, especially in older material, but current Globus documentation distinguishes an **endpoint** (the server/system) from a **collection** (the data-access location actually used for transfers). [Globus Docs](https://docs.globus.org/guides/overviews/collections-and-endpoints/?utm_source=chatgpt.com)

Write a step-by-step procedure that would transfer a dataset from one location to another using Globus.
Here is a clear step-by-step procedure you could use for an assignment:

1. **Log in to Globus.**
   Open the Globus web application and sign in using your university or research institution account.

2. **Open File Manager.**
   In Globus, go to **File Manager**. This is where you choose the source and destination locations.

3. **Select the source collection.**
   Search for the Globus collection that contains the dataset. For example:

   ```text
   Source collection: My Laptop
   Source folder: /Research/GAMEEMO/
   ```

   If the files are on a personal computer, Globus Connect Personal must be installed and running.

4. **Select the destination collection.**
   Open the second File Manager panel and search for the destination system, such as a university HPC cluster.

   ```text
   Destination collection: University HPC Storage
   Destination folder: /home/user/EEG_Data/
   ```

5. **Choose the dataset to transfer.**
   On the source side, select the folder or files you want to move. For example:

   ```text
   GAMEEMO/
   ├── S01/
   ├── S02/
   ├── S03/
   └── ...
   ```

6. **Choose the destination directory.**
   On the destination side, navigate to the directory where the dataset should be stored:

   ```text
   /home/user/EEG_Data/
   ```

7. **Start the transfer.**
   Click **Start** or the transfer arrow in Globus.

   The requested transfer would conceptually be:

   ```text
   My-Laptop:/Research/GAMEEMO/
              ↓
           Globus
              ↓
   University-HPC:/home/user/EEG_Data/
   ```

8. **Monitor the transfer.**
   Open the **Activity** page in Globus. The transfer may initially appear as:

   ```text
   Status: ACTIVE
   ```

   Globus will manage the transfer, retry temporary failures, and check whether the files were transferred successfully.

9. **Confirm that the transfer completed.**
   When the transfer finishes, the status should show:

   ```text
   Status: SUCCEEDED
   ```

10. **Verify the files on the destination system.**
    If the destination is a cluster, log into it and check that the files are present:

   ```bash
   cd /home/user/EEG_Data/
   ls
   ```

   You should then see the transferred dataset.

In short, the procedure is:

```text
Log into Globus
      ↓
Open File Manager
      ↓
Choose source collection
      ↓
Choose destination collection
      ↓
Select dataset
      ↓
Select destination folder
      ↓
Start transfer
      ↓
Monitor Activity
      ↓
Verify successful transfer
```

The main advantage is that after you start the transfer, **Globus manages the transfer for you instead of requiring you to keep a manual file-copy command running continuously.**

### 🧭 Activity A.4 — The Compute Access Model Ladder: From Manual Sessions to Serverless

There's a second, distinct ladder hiding inside everything you've done so far — not *how powerful* the hardware is (that's the CI-Hardware ladder), but *how you actually get time on it*:

`manual/interactive session → batch/scheduled jobs (Slurm, what you just learned) → persistent request-hold-release testbed sessions (FABRIC, Colabs 1-3) → serverless/on-demand compute (the emerging model)`

**A real number worth knowing:** as of 2026, dedicated/reserved GPU cloud capacity costs roughly $2.65/GPU-hour on a one-year commitment — but serverless GPU platforms bill only for active compute time and scale to zero between jobs, which is why the industry moved toward serverless as the dominant paradigm this year: most research and inference workloads are bursty, not constant, and paying for idle reserved capacity wastes money on exactly the pattern FABRIC's own "request-hold-release" model was designed around.

**Ask your AI tool:** "Compare a persistent request-hold-release compute session (like FABRIC) to a serverless, scale-to-zero compute model, for a workload that runs unpredictably — a few times a day, for a few minutes each time. Which is more cost-effective, and why?"

In [ ]:
# 📝 Activity A.4 — Paste your AI conversation and your own short answer here:
For a workload that runs **only a few times per day for a few minutes at a time**, a **serverless, scale-to-zero model is usually more cost-effective**.

With a persistent request-hold-release model like a reserved research compute session, you typically acquire resources, keep them allocated while you work, and then release them. The key inefficiency is **idle time**: if you hold a VM or node for several hours but actually compute for only 10–20 minutes total, most of that allocation sits unused.

By contrast, serverless compute is designed for bursty workloads. It can scale from zero when a job arrives, run for a short period, and scale back to zero afterward. You generally pay only for the resources consumed while the code is actually executing.

For example, suppose your analysis runs 4 times per day for 5 minutes each:

| Model | Active compute | Idle resources |
|---|---:|---:|
| Persistent session held for 8 hours | 20 min | ~7 hr 40 min |
| Serverless scale-to-zero | ~20 min | ~0 min |

So serverless is a much better fit when the workload is **short, unpredictable, and intermittent**.

A persistent model can still make more sense when you need specialized hardware, custom networking, multiple coordinated nodes, large local datasets, or a research environment that must stay configured between runs. FABRIC, for example, is especially useful when the experiment involves **network topology, distributed systems, or controlled research infrastructure**, rather than simply executing short independent functions.

So the rule of thumb is:

**Unpredictable + short + infrequent → serverless is usually more cost-efficient.**

**Long-running + stateful + specialized infrastructure → persistent allocated compute is usually more appropriate.**

One caveat: on an academic testbed such as FABRIC, “cost” may mean **allocation/resource efficiency rather than a direct per-minute cloud bill**. Even then, releasing unused nodes promptly is important because those resources are shared with other researchers.

> **Add to your Ladder document:** your Compute Access Model rungs, citing the $2.65/GPU-hour figure or a number you find yourself researching serverless GPU pricing.

### ✍️ Part A Reflection



*Double-click this cell to write your response.*







* **What I observed:** What information does a `#SBATCH` header communicate to the scheduler that you didn't have to think about when running code locally?
***
**What #Sbatch does is provide resource requirements to the scheduler so it is able to properly allocate resources like RAM and cores for your project/job, something that is done automatically on a local device.**
***

* **Connecting to key concepts:** Connects to the new **CI Software & Resource Management track**: Job Scheduler, Resource Request Directives, Job State, Shared Cluster Etiquette. Explain why "request only what you need, and release it promptly" (FABRIC Concept 14) applies just as much to batch jobs as to interactive sessions.

***
**Request only what you need, and release it promptly is simply just the polite thing to do so that you can minimize resources for yourself and not use unnecessary resources that others could be using for their research or projects**
***

* **On Activity A.3 (self-directed exploration):** What was different about teaching yourself Globus with AI, compared to Activities A.1 and A.2 where a prompt was already written for you? What did you have to double-check because the AI's answer sounded plausible but wasn't obviously verified?
***
**I had to do some double checking with the prompts, using the website provided and doing my own reasearch to make sure the AI wasn't generating false data or explinations**





***



## 💻 Part B (Build 1 of 2): Multi-Node Distributed Circuit Model



### Topic: Real Multi-Node Parallelism







### 🧭 Activity B.1 — Distributing the Circuit Model Across Multiple Nodes







> "Take the brain-circuit model from Colab 1 (or its cohort-scaled version from Colab 2). Write code that distributes it across multiple simulated FABRIC nodes — each node handling a subset of participants or circuit instances — and combines the results at the end. Add comments explaining what has to be communicated between nodes versus what each node can compute independently."

In [ ]:
# 💻 Activity B.1 — Paste your AI-generated multi-node distribution code here:

### 🧭 Activity B.2 — Adding a Job Dependency







> "Extend Activity B.1 so that a final 'combine results' job only starts after all the individual node jobs have completed successfully — a job dependency. Show what happens (in your code's logic) if one node's job fails."

In [ ]:
# 💻 Activity B.2 — Paste your AI-generated job-dependency code here:

### ✍️ Part B Reflection



*Double-click this cell to write your response.*







* **What I observed:** What was the communication overhead — the time spent distributing data and collecting results — relative to the actual computation time?



* **Connecting to key concepts:** Connects to **Certification Concept 27** (parallel bottleneck) and the new track's **Multi-Node Parallelism** and **Job Dependency** concepts.







***



> ## ✅ Colab 3-1 Submission Checkpoint (due end of Week 5)



> Submit through Activity B.2 and its reflection — the Slurm/resource-management Primer and the multi-node circuit-model build.







***



## 🔌 Part C (Build 2 of 2): Two-Way Edge-Device Flow



### Topic: Beyond One-Way Output — Edge Devices That Receive, Too







This is the course's first micro:bit workflow. Real edge devices like the micro:bit have only 128 KB of RAM and no `numpy`/`scipy` — so before adding a two-way flow, you'll first build a one-way sensor-reading workflow that respects that limit, then extend it to receive as well as send.







### 🧭 Activity C.1a — RAM-Checked Sensor Reading







> "Write a Python class simulating a micro:bit reading a live sensor stream one sample at a time into a buffer, checking available RAM before each read. If the buffer would exceed a 128 KB limit, print `[ERROR]: Out of RAM!` and stop; otherwise print `[SUCCESS] Used RAM: ... bytes.` once the full stream has been read. Use only basic arithmetic — no numpy or scipy."

In [ ]:
# 💻 Activity C.1a — Paste your AI-generated RAM-checked sensor-reading code here:





### 🧭 Activity C.1b — Sending a Command Back to the Edge Device







> "Extend the RAM-checked sensor-reading workflow from Activity C.1a so the device can also *receive* a signal — for example, a calibration value or a mode switch — from the rest of the workflow, and change its behavior in response. Explain in comments how this differs from the one-way output-only version."

In [ ]:
# 💻 Activity C.1b — Paste your AI-generated two-way edge-device code here:

### 🧭 Activity C.2 — A Task of Your Team's Choosing







> "Using the two-way flow from Activity C.1, design and implement a small task where the incoming signal actually changes what the device does next (not just displays a value). Pick something your team finds interesting — e.g., adjusting a detection threshold in real time based on an external signal. Document your design choice in a comment before writing the code."

In [ ]:
# 💻 Activity C.2 — Paste your AI-generated team-chosen two-way task code here:

### ✍️ Part C Reflection

*Double-click this cell to write your response.*



* **What I observed:** What did you choose to build, and why? What would break if the "receive" half of the flow failed silently?

* **Connecting to key concepts:** Connects to Track B's **[Latency in Closed-Loop Control](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackB11)** applied to a two-way link, and the new track's **[Job State](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA8)** concept — analogous to knowing whether the edge device's last received command was actually applied.



***

> ## ✅ Colab 3-2 Submission Checkpoint (due end of Week 6, combined)

> Submit the complete notebook (Parts A, B, and C, all reflections). This is the full Colab 3 deliverable and earns the advanced-neuro-workflows/resource-management microcredential.

>

> **This week, your team also selects its capstone project** from the faculty-sourced pool (see the Capstone Project Template).



***

## 📑 What Comes Next



Colab 4 introduces the Basics of AI, Agentic AI, and Physical AI, and reuses the multi-node circuit model you built here as the "control" half of a semi-realistic brain network for movement.

***
## 🎯 Foundational Concepts: This Colab's Microcredential Targets

This section is the actual outcome of Colab 3's microcredential — not an afterthought at the end. Below are the 11 core concepts this Colab is built to teach: Job Scheduler, Job Queue, Wall-Clock Time Limit, Job State, Resource Request Directives, Multi-Node Parallelism, Job Dependency, Shared Cluster Etiquette, Community Resource Ethics, Latency in Closed-Loop Control, and the Compute Access Model Ladder.

For each one: read the short resource, run the code if there is any, then **go find and answer a real practice question about it from somewhere other than this course** — a textbook companion site, a university's open courseware quiz, a MOOC's practice set, anywhere. These 11 concepts are exactly what the Target Questions at the end of this Colab — and later, the quizzes and Final — will test.

### Concept 1 — Job Scheduler



**Resource:** A job scheduler (like Slurm) decides which queued jobs run where and when on a shared cluster, based on requested resources and priority — the shared-computing equivalent of a host seating people at a busy restaurant.

**🔍 Find-and-Practice:** Search the web for a practice question about *Slurm or job scheduling on HPC clusters* from a source that isn't this course. Paste the question, your answer, and the source URL below.

In [ ]:
# 📝 Paste your found question, your answer, and the source URL here:

### Concept 2 — Job Queue



**Resource:** The job queue is the ordered waiting line of submitted jobs, held until their requested resources become free — your job doesn't start the instant you submit it, it starts once the scheduler has room.

In [ ]:
queue = ["job_A (4 nodes)", "job_B (2 nodes)", "job_C (8 nodes)"]

available_nodes = 6

print("Next job that can start:", queue[1], "-- job_A and job_C must wait for more nodes to free up.")

**🔍 Find-and-Practice:** Search the web for a practice question about *HPC job queues* from a source that isn't this course. Paste the question, your answer, and the source URL below.

In [ ]:
# 📝 Paste your found question, your answer, and the source URL here:

### Concept 3 — Wall-Clock Time Limit



**Resource:** A wall-clock time limit is the maximum real-world runtime a job is allowed before the scheduler kills it, no matter what state it's in — this protects shared resources from being monopolized by one runaway job.

**🔍 Find-and-Practice:** Search the web for a practice question about *wall-clock limits in batch computing* from a source that isn't this course. Paste the question, your answer, and the source URL below.

In [ ]:
# 📝 Paste your found question, your answer, and the source URL here:

### Concept 4 — Job State



**Resource:** A submitted job moves through states — Pending, Running, Completed, Failed — and checking that state is how you know whether to keep waiting, start using your results, or start debugging.

In [ ]:
job_states = {"job_A": "RUNNING", "job_B": "PENDING", "job_C": "COMPLETED"}

for job, state in job_states.items():

    print(f"{job}: {state}")

**🔍 Find-and-Practice:** Search the web for a practice question about *job state monitoring in Slurm (squeue)* from a source that isn't this course. Paste the question, your answer, and the source URL below.

In [ ]:
# 📝 Paste your found question, your answer, and the source URL here:

### Concept 5 — Resource Request Directives



**Resource:** Directives like `#SBATCH --nodes=4` tell the scheduler up front exactly how many nodes, cores, and how much memory and time a job needs — without this, the scheduler has no way to plan around your job.

**🔍 Find-and-Practice:** Search the web for a practice question about *#SBATCH directives or resource requests* from a source that isn't this course. Paste the question, your answer, and the source URL below.

In [ ]:
# 📝 Paste your found question, your answer, and the source URL here:

### Concept 6 — Multi-Node Parallelism



**Resource:** Splitting one computation across several separate machines (nodes) that each work independently, then combining results, is different from just using multiple cores on one machine — it requires explicitly deciding what needs to be communicated between nodes.

**🔍 Find-and-Practice:** Search the web for a practice question about *multi-node parallel computing* from a source that isn't this course. Paste the question, your answer, and the source URL below.

In [ ]:
# 📝 Paste your found question, your answer, and the source URL here:

### Concept 7 — Job Dependency



**Resource:** Configuring one job to wait until another finishes successfully before starting — e.g., don't combine results until every node's individual job has completed — prevents a workflow from silently working with incomplete data.

In [ ]:
node_statuses = ["COMPLETED", "COMPLETED", "FAILED"]

if all(s == "COMPLETED" for s in node_statuses):

    print("Safe to combine results.")

else:

    print("Blocked: not all nodes completed successfully.")

**🔍 Find-and-Practice:** Search the web for a practice question about *job dependencies in Slurm* from a source that isn't this course. Paste the question, your answer, and the source URL below.

In [ ]:
# 📝 Paste your found question, your answer, and the source URL here:

### Concept 8 — Shared Cluster Etiquette



**Resource:** On a shared research computing system, requesting only the resources you actually need — and releasing them promptly when done — isn't just politeness, it's what keeps the system usable for everyone else waiting in the queue.

**🔍 Find-and-Practice:** Search the web for a practice question about *responsible use of shared research computing resources* from a source that isn't this course. Paste the question, your answer, and the source URL below.

In [ ]:
# 📝 Paste your found question, your answer, and the source URL here:

### Concept 9 — Community Resource Ethics



**Resource:** National computing systems like FABRIC are funded to serve an entire research community, not any one lab — the same "request only what you need" principle from shared cluster etiquette applies at the level of an entire national testbed.

**🔍 Find-and-Practice:** Search the web for a practice question about *research computing resource allocation policy* from a source that isn't this course. Paste the question, your answer, and the source URL below.

In [ ]:
# 📝 Paste your found question, your answer, and the source URL here:

### Concept 10 — Latency in Closed-Loop Control



**Resource:** In a system that senses, decides, and acts in a loop, the delay between sensing and acting is latency — too much of it can make the system unstable or unsafe, since it ends up reacting to information that's no longer current.

**🔍 Find-and-Practice:** Search the web for a practice question about *latency in real-time or closed-loop control systems* from a source that isn't this course. Paste the question, your answer, and the source URL below.

In [ ]:
# 📝 Paste your found question, your answer, and the source URL here:

### Concept 11 — Compute Access Model Ladder

**Resource:** Manual sessions → batch/Slurm → persistent testbed sessions (FABRIC) → serverless/on-demand. Real number: reserved GPU cloud capacity runs roughly $2.65/GPU-hour on a one-year commitment, while serverless platforms bill only for active use and scale to zero between jobs — why the industry shifted toward serverless as the dominant 2026 model for bursty workloads.

In [ ]:
# 🔍 Find-and-Practice: search for a practice question about serverless computing or cloud cost models from an external source. Paste it, your answer, and the source URL here:

***
## 🎯 Target Questions

These are the questions the 10 concepts above prepare you for. They're the same concepts — and often close to the same format — as what appears on the quizzes and Final. **Each question has exactly TWO correct answers**, matching the actual quiz format.

No answer key is provided here on purpose. Work out your answer, then use an AI tool to check your reasoning and confirm or correct it — that verification step is itself part of the skill this course is built around.

**1.** Which TWO describe what a job scheduler does?
A) Decides which queued jobs run where and when.
B) Makes allocation decisions based on requested resources and priority.
C) Guarantees every job starts the instant it's submitted.
D) Only manages jobs from one specific user.
E) Replaces the need for any resource request information.

**2.** Which TWO describe a job queue?
A) The ordered waiting line of submitted jobs.
B) Jobs wait here until their requested resources become free.
C) It guarantees jobs run in the order submitted regardless of resource needs.
D) It's a backup system that saves job results automatically.
E) It only exists on personal laptops.

**3.** Which TWO describe a wall-clock time limit?
A) The maximum real-world runtime a job is allowed before being killed.
B) It protects shared resources from being monopolized by one job.
C) It measures the code's quality.
D) It only applies to GPU jobs.
E) It is set automatically with no researcher input ever required.

**4.** Which TWO describe job state monitoring?
A) A job can be Pending, Running, Completed, or Failed.
B) Checking state tells you whether to keep waiting or start debugging.
C) A job's state never changes once submitted.
D) Only completed jobs have a state.
E) Job state is unrelated to whether you should trust the results yet.

**5.** Which TWO describe resource request directives (e.g., #SBATCH)?
A) They specify resources like nodes, cores, memory, and time limit.
B) The scheduler reads them before deciding when/where to run the job.
C) They control the visual theme of output plots.
D) They determine which programming language the script uses.
E) They are optional and have no effect on scheduling.

**6.** Which TWO describe multi-node parallelism?
A) A computation is split across several separate machines.
B) What's communicated between nodes must be explicitly decided.
C) It's identical to using multiple cores on a single machine.
D) It never requires combining results afterward.
E) It only works with exactly two nodes.

**7.** Which TWO describe a job dependency?
A) One job waits until another finishes successfully before starting.
B) It prevents a workflow from combining incomplete results.
C) It means all jobs must run at the exact same time.
D) It has no effect on job scheduling order.
E) It only applies to single-node jobs.

**8.** Which TWO reflect good shared-cluster etiquette?
A) Requesting only the resources you actually need.
B) Releasing resources promptly when done.
C) Requesting the maximum possible resources "just in case."
D) Leaving idle sessions running indefinitely.
E) Ignoring queue wait times for other users.

**9.** Which TWO describe community resource ethics on a system like FABRIC?
A) The system is funded to serve an entire research community.
B) The "request only what you need" principle applies at a national scale.
C) Individual labs own dedicated permanent allocations.
D) Ethics only apply to paid commercial cloud systems.
E) There is no shared responsibility among users.

**10.** Which TWO describe latency in closed-loop control?
A) It's the delay between sensing and acting.
B) Too much latency can make a system unstable or unsafe.
C) Latency doesn't matter as long as final accuracy is high.
D) Latency only affects how results are visualized.
E) Latency cannot be influenced by system design choices.

**11.** Which TWO correctly describe the Compute Access Model ladder and its economics?

A. It runs from manual sessions, to batch/Slurm jobs, to persistent testbed sessions, to serverless/on-demand compute.
B. Serverless GPU platforms bill only for active compute time and scale to zero between jobs.
C. Reserved GPU cloud capacity and serverless GPU capacity always cost exactly the same per hour.
D. FABRIC's request-hold-release model is identical to a serverless, scale-to-zero model.
E. Serverless compute requires a permanent, unreleasable session to be requested in advance.

***

## ✍️ Now It's Your Turn: Write 10 Target Questions of Your Own



You've worked through 10 concept resources, hunted down external practice questions for each, and answered the 10 official Target Questions above. Now flip the process: write **10 of your own** target-style questions covering this Colab's material.



Your questions can be:

- Harder or more specific versions of the concepts above,

- Questions about something this Colab covered that wasn't in the official 10,

- Questions inspired by whatever you found during your Find-and-Practice research.



Use the same two-answer-select format (5 options, exactly 2 correct) or a short conceptual question — whichever fits the idea better. Include the correct answer(s) and a one-sentence explanation for each.



**Why this matters:** writing a good test question requires understanding the material more deeply than just answering one — you have to know not just what's true, but what's *plausibly wrong* (the distractors). This is also genuinely useful beyond your own studying: your best questions may get folded into a future semester's quiz bank, the same way the Advanced Explorations Library's Field-Scan Challenge (Option E) uses student proposals to keep that material current.

In [ ]:
# ✍️ Write your 10 original questions, answers, and explanations here:

***
## 📖 Reference Guide: Concepts Used in Colab 3

**From the master 50:** [Concept 27](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#q27) (Parallel Bottleneck).

**From Track B:** [Latency in Closed-Loop Control](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackB11) (corrected — "processing delay" isn't in the master 50; it's this Track B concept).

**From FABRIC's 19:** [Concept 14](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#fabric14) (Community Resource Ethics).

**From the new CI Software & Resource Management track:** [Job Scheduler](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA1), [Batch Job](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA2), [Job Queue](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA3), [Node Allocation](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA4), [Wall-Clock Time Limit](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA6), [Job State](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA8), [Resource Request Directives](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA9), [Multi-Node Parallelism](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA10), [Job Dependency](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA12), [Shared Cluster Etiquette](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA15).

**New platform explored via self-directed vibe coding:** Globus (research data transfer).

**Technique adopted from the Workshop's current Lab 3:** RAM-checked buffered sensor reading (Activity C.1a) — reading one sample at a time into a bounded buffer, checking available memory before each read.

***
## ✅ Self-Check: What Did You Learn?

| Concept Area | Concept | Where You Demonstrated It |
|---|---|---|
| CI Software | [Job Scheduler](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA1) / #SBATCH directives | [ Enter Activity ID ] |
| CI Software | [Job State](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA8) (Pending/Running/Completed/Failed) | [ Enter Activity ID ] |
| CI Software | [Multi-Node Parallelism](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA10) | [ Enter Activity ID ] |
| CI Software | [Job Dependency](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA12) | [ Enter Activity ID ] |
| CI Software | [Shared Cluster Etiquette](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#trackA15) | [ Enter Activity ID ] |
| Computing | Concept 27 — Parallel bottleneck | [ Enter Activity ID ] |
| CI Software | Self-directed platform exploration (Globus) | [ Enter Activity ID ] |

**From the Cumulative Microcredential Ladders:** [Compute Access Model Ladder](https://019fd585-bab7-9f63-bb09-38e93ede654b.share.connect.posit.cloud/certification.html#ladder5).